# Stage A — Process raw participant data

Reads the per-subject jsPsych CSVs produced by the listener experiment
([`web/cogsci26_rsa_listener_experiment_n5_o1/`](../../web/cogsci26_rsa_listener_experiment_n5_o1/)),
pivots them to one row per participant, and emits two public artifacts:

1. `processed_listener_n1_anonymized.csv` — wide-form per-participant table
   with PII columns dropped.
2. `speaker_type.csv` — aggregate proportions of speaker-type judgments per
   `(speaker_sequence, round, speaker_type)` over the vigilant-condition
   participants. This is the table consumed by `fit_speaker_type.ipynb`
   (originally produced by an R script in the source repo; reimplemented
   here so the pipeline is fully Python).

The intermediate file with identifiers (`processed_listener_n1_full.csv`)
stays under `raw_do_not_track/` and is never committed.

## Imports & helpers

In [ ]:
import os, sys, json
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime


def parse_json_response(s):
    """Parse jsPsych JSON-encoded response field."""
    if pd.isna(s) or s == '':
        return {}
    try:
        return json.loads(s) if isinstance(s, str) else s
    except (json.JSONDecodeError, TypeError):
        return {}


## Per-participant extraction

In [ ]:
def process_single_participant(filepath):
    """
    Process a single participant's CSV file and return a dictionary
    with all relevant variables in wide format.
    
    Handles both old and new data formats (with/without datapipe_condition, cell_idx, etc.)
    """
    try:
        df = pd.read_csv(filepath)
    except Exception as e:
        print(f"Error reading {filepath}: {e}")
        return None
    
    if len(df) == 0:
        return None
    
    result = {}
    
    # =========================================================================
    # METADATA (from first row with completion info)
    # =========================================================================
    meta_cols = [
        'subject_id', 'prolific_pid', 'study_id', 'session_id',
        'experiment_version', 'start_time', 'completion_status', 
        'completion_time', 'terminated_early', 'termination_reason',
        'speaker_condition', 'listener_belief_condition', 
        'sequence_idx', 'measure_order',
        # New columns in updated experiment
        'datapipe_condition', 'cell_idx'
    ]
    
    for col in meta_cols:
        if col in df.columns:
            # Get first non-empty value (excluding 'null' strings)
            non_empty = df[df[col].notna() & (df[col] != '') & (df[col] != 'null')][col]
            result[col] = non_empty.iloc[0] if len(non_empty) > 0 else None
    
    # Calculate total experiment duration
    if 'time_elapsed' in df.columns:
        result['total_duration_ms'] = df['time_elapsed'].max()
        result['total_duration_min'] = result['total_duration_ms'] / 60000 if result['total_duration_ms'] else None
    
    # =========================================================================
    # COMPREHENSION CHECKS
    # =========================================================================
    
    # comp1_some - "some" definition check
    comp1_some = df[df['task'] == 'comp1_some']
    if len(comp1_some) > 0:
        result['comp1_some_correct'] = comp1_some['comp1_some_correct'].iloc[0]
        # Get their response
        resp = parse_json_response(comp1_some['response'].iloc[0])
        result['comp1_some_response'] = resp.get('some_def', None)
    
    # comp1_most - "most" definition check
    comp1_most = df[df['task'] == 'comp1_most']
    if len(comp1_most) > 0:
        result['comp1_most_correct'] = comp1_most['comp1_most_correct'].iloc[0]
        resp = parse_json_response(comp1_most['response'].iloc[0])
        result['comp1_most_response'] = resp.get('most_def', None)
    
    # comp2_tf - True/False check (single image)
    comp2 = df[df['task'] == 'comp2_tf']
    if len(comp2) > 0:
        result['comp2_correct'] = comp2['comp2_correct'].iloc[0]
    
    # comp_multiple_desc - Multiple descriptions check
    comp_multi = df[df['task'] == 'comp_multiple_desc']
    if len(comp_multi) > 0:
        result['comp_multi_correct'] = comp_multi['comp_correct'].iloc[0]
        result['comp_multi_selected'] = comp_multi['selected'].iloc[0]
    
    # comp3 - Which trials make statement true
    comp3 = df[df['task'] == 'comp3']
    if len(comp3) > 0:
        result['comp3_correct'] = comp3['comp3_correct'].iloc[0]
        result['comp3_selected'] = comp3['selected'].iloc[0]
    
    # Truth comprehension check
    truth_comp = df[df['task'] == 'truth_comprehension']
    if len(truth_comp) > 0:
        result['truth_comp_correct'] = truth_comp.get('truth_check_correct', pd.Series([None])).iloc[0]
    
    # =========================================================================
    # MAIN TRIAL DATA (rounds 1-5)
    # =========================================================================
    
    # Combined measure (vigilant condition) or point_estimate_effectiveness (other conditions)
    trial_tasks = ['combined_measure', 'point_estimate_effectiveness']
    trials = df[df['task'].isin(trial_tasks)]
    
    for _, row in trials.iterrows():
        round_num = row.get('round', None)
        if pd.isna(round_num) or round_num == '':
            continue
        round_num = int(round_num)
        
        prefix = f'r{round_num}'
        
        # Utterance info
        result[f'{prefix}_utterance_predicate'] = row.get('utterance_predicate', None)
        result[f'{prefix}_utterance_quantifier'] = row.get('utterance_quantifier', None)
        result[f'{prefix}_utterance_text'] = row.get('utterance_text', None)
        
        # Responses
        result[f'{prefix}_effectiveness'] = row.get('effectiveness_point_estimate', None)
        
        # Speaker type (may be empty for non-vigilant conditions)
        speaker_type = row.get('speaker_type_point_estimate', None)
        if speaker_type and speaker_type != '' and speaker_type != 'null':
            result[f'{prefix}_speaker_type'] = speaker_type
        else:
            result[f'{prefix}_speaker_type'] = None
        
        # RT (if available)
        if 'rt' in row and row['rt'] not in [None, '', 'null']:
            try:
                result[f'{prefix}_rt'] = float(row['rt'])
            except (ValueError, TypeError):
                result[f'{prefix}_rt'] = None
    
    # Separate speaker goal page (if exists, for older version)
    speaker_goal_trials = df[df['task'] == 'point_estimate_speaker_goal']
    for _, row in speaker_goal_trials.iterrows():
        round_num = row.get('round', None)
        if pd.isna(round_num) or round_num == '':
            continue
        round_num = int(round_num)
        prefix = f'r{round_num}'
        
        # Only fill if not already filled by combined_measure
        if f'{prefix}_speaker_type' not in result or result[f'{prefix}_speaker_type'] is None:
            result[f'{prefix}_speaker_type'] = row.get('speaker_type_point_estimate', None)
    
    # =========================================================================
    # ATTENTION CHECK
    # =========================================================================
    
    attention = df[df['task'] == 'attention_check']
    if len(attention) > 0:
        row = attention.iloc[0]
        result['attention_check_passed'] = row.get('attention_check_passed', None)
        result['attention_effectiveness'] = row.get('effectiveness_point_estimate', None)
        
        # Speaker type may be null for non-vigilant conditions
        speaker_type = row.get('speaker_type_point_estimate', None)
        if speaker_type and speaker_type != '' and speaker_type != 'null':
            result['attention_speaker_type'] = speaker_type
        else:
            result['attention_speaker_type'] = None
    
    # =========================================================================
    # COMPETENCE RATING
    # =========================================================================
    
    competence = df[df['task'] == 'competence_rating']
    if len(competence) > 0:
        row = competence.iloc[0]
        result['competence_score'] = row.get('competence_score', None)
        result['competence_framing'] = row.get('competence_framing', None)
        
        # NEW: estimation_strategy may now be stored directly in this row
        if 'estimation_strategy' in row and pd.notna(row['estimation_strategy']) and row['estimation_strategy'] != '':
            result['open_estimation_strategy'] = row['estimation_strategy']
    
    # =========================================================================
    # PERSUASIVE SPEAKER REVEAL (vigilant + persuasive only)
    # =========================================================================
    
    reveal = df[df['task'] == 'persuasive_speaker_reveal']
    result['saw_persuasive_reveal'] = len(reveal) > 0
    
    # =========================================================================
    # OPEN-ENDED RESPONSES
    # =========================================================================
    
    survey_text = df[df['trial_type'] == 'survey-text']
    for _, row in survey_text.iterrows():
        resp = parse_json_response(row.get('response', ''))
        
        # Vigilant condition questions
        if 'speaker_evaluation_strategy' in resp:
            result['open_speaker_strategy'] = resp['speaker_evaluation_strategy']
        
        # Other conditions question (old format where it's in survey-text)
        if 'estimation_strategy' in resp:
            # Only set if not already set from competence_rating row
            if 'open_estimation_strategy' not in result or result['open_estimation_strategy'] is None:
                result['open_estimation_strategy'] = resp['estimation_strategy']
        
        # General feedback (all conditions)
        if 'feedback' in resp:
            result['open_feedback'] = resp['feedback']
    
    # =========================================================================
    # DERIVED VARIABLES
    # =========================================================================
    
    # Count how many rounds completed
    completed_rounds = sum(1 for i in range(1, 6) 
                          if result.get(f'r{i}_effectiveness') is not None)
    result['n_rounds_completed'] = completed_rounds
    
    # Did they complete all rounds?
    result['completed_all_rounds'] = completed_rounds == 5
    
    # Calculate comprehension score (out of 5 checks)
    comp_checks = [
        result.get('comp1_some_correct'),
        result.get('comp1_most_correct'),
        result.get('comp2_correct'),
        result.get('comp_multi_correct'),
        result.get('comp3_correct')
    ]
    result['n_comp_correct'] = sum(1 for c in comp_checks if c == True or c == 'true')
    
    return result



## Batch helpers

In [ ]:
def process_folder(input_folder, output_file=None):
    """
    Process all CSV files in a folder and combine into a single DataFrame.
    """
    input_path = Path(input_folder)
    
    if not input_path.exists():
        print(f"Error: Input folder '{input_folder}' does not exist.")
        return None
    
    # Find all CSV files
    csv_files = list(input_path.glob('*.csv'))
    
    if len(csv_files) == 0:
        print(f"Error: No CSV files found in '{input_folder}'")
        return None
    
    print(f"Found {len(csv_files)} CSV files to process...")
    
    # Process each file
    all_results = []
    errors = []
    
    for i, filepath in enumerate(csv_files):
        if (i + 1) % 10 == 0:
            print(f"  Processing {i + 1}/{len(csv_files)}...")
        
        result = process_single_participant(filepath)
        if result is not None:
            result['source_file'] = filepath.name
            all_results.append(result)
        else:
            errors.append(filepath.name)
    
    print(f"\nSuccessfully processed: {len(all_results)} participants")
    if errors:
        print(f"Errors: {len(errors)} files")
        for err in errors[:5]:
            print(f"  - {err}")
        if len(errors) > 5:
            print(f"  ... and {len(errors) - 5} more")
    
    # Create DataFrame
    df = pd.DataFrame(all_results)
    
    # Reorder columns for readability
    col_order = [
        # Metadata
        'source_file', 'subject_id', 'prolific_pid', 'study_id', 'session_id',
        'experiment_version', 'start_time', 'completion_status', 'completion_time',
        'terminated_early', 'termination_reason', 'total_duration_ms', 'total_duration_min',
        # New metadata columns
        'datapipe_condition', 'cell_idx',
        
        # Conditions
        'speaker_condition', 'listener_belief_condition', 'sequence_idx', 'measure_order',
        
        # Comprehension
        'comp1_some_correct', 'comp1_some_response',
        'comp1_most_correct', 'comp1_most_response',
        'comp2_correct', 'comp_multi_correct', 'comp_multi_selected',
        'comp3_correct', 'comp3_selected', 'truth_comp_correct', 'n_comp_correct',
        
        # Rounds 1-5
        'r1_utterance_predicate', 'r1_utterance_quantifier', 'r1_utterance_text',
        'r1_effectiveness', 'r1_speaker_type', 'r1_rt',
        'r2_utterance_predicate', 'r2_utterance_quantifier', 'r2_utterance_text',
        'r2_effectiveness', 'r2_speaker_type', 'r2_rt',
        'r3_utterance_predicate', 'r3_utterance_quantifier', 'r3_utterance_text',
        'r3_effectiveness', 'r3_speaker_type', 'r3_rt',
        'r4_utterance_predicate', 'r4_utterance_quantifier', 'r4_utterance_text',
        'r4_effectiveness', 'r4_speaker_type', 'r4_rt',
        'r5_utterance_predicate', 'r5_utterance_quantifier', 'r5_utterance_text',
        'r5_effectiveness', 'r5_speaker_type', 'r5_rt',
        
        # Completion
        'n_rounds_completed', 'completed_all_rounds',
        
        # Attention check
        'attention_check_passed', 'attention_effectiveness', 'attention_speaker_type',
        
        # Final measures
        'competence_score', 'competence_framing', 'saw_persuasive_reveal',
        
        # Open-ended
        'open_speaker_strategy', 'open_estimation_strategy', 'open_feedback',
    ]
    
    # Only include columns that exist
    existing_cols = [c for c in col_order if c in df.columns]
    other_cols = [c for c in df.columns if c not in col_order]
    df = df[existing_cols + other_cols]
    
    # Save if output file specified
    if output_file:
        df.to_csv(output_file, index=False)
        print(f"\nSaved to: {output_file}")
    
    return df



def add_participant_id(df):
    """Add participant_id column (P001, P002, etc.) at the beginning."""
    df = df.copy()
    df.insert(0, 'participant_id', [f'P{i+1:03d}' for i in range(len(df))])
    return df

def create_anonymized_version(df):
    """
    Create anonymized version by removing identifying columns.
    Keeps participant_id as the only identifier.
    """
    # Columns to remove for anonymization
    id_columns = ['source_file', 'subject_id', 'prolific_pid', 'study_id', 'session_id']
    
    df_anon = df.copy()
    for col in id_columns:
        if col in df_anon.columns:
            df_anon = df_anon.drop(columns=[col])
    
    return df_anon

## Run the pipeline

Edit `INPUT_FOLDER` if your raw CSVs live elsewhere.

In [ ]:
# === EDIT THESE PATHS IF YOUR RAW DATA IS ELSEWHERE ===
INPUT_FOLDER = './raw_do_not_track/prag_net_listener_n1_main'
OUTPUT_FULL = './raw_do_not_track/processed_listener_n1_full.csv'   # contains identifiers
OUTPUT_ANON = './processed_listener_n1_anonymized.csv'              # public artifact

# Stage A.1 — process raw subject CSVs into one wide-form row per participant.
df = process_folder(INPUT_FOLDER, output_file=None)
df = add_participant_id(df)

# Save the full version (still contains identifiers — kept under raw_do_not_track).
Path(OUTPUT_FULL).parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_FULL, index=False)
print(f"Saved full processed data to: {OUTPUT_FULL}")
print(f"  Rows: {len(df)}  Columns: {len(df.columns)}")

# Stage A.2 — anonymize. Strips identifying columns and the start/completion
# timestamps (quasi-identifying) before writing the public artifact.
df_anon = create_anonymized_version(df)
for ts_col in ('start_time', 'completion_time'):
    if ts_col in df_anon.columns:
        df_anon = df_anon.drop(columns=[ts_col])

df_anon.to_csv(OUTPUT_ANON, index=False)
print(f"\nSaved anonymized data to: {OUTPUT_ANON}")
print(f"  Rows: {len(df_anon)}  Columns: {len(df_anon.columns)}")


## Aggregate `speaker_type.csv`

Vigilant-condition participants pick a `speaker_type ∈ {anti, neutral, pro}`
on every round. The downstream `fit_speaker_type.ipynb` expects the data
aggregated as one row per (`speaker_sequence`, `round`, `speaker_type`) with
counts and percentages.

Replaces the R-Markdown's `dplyr::count` / `complete` pipeline.

In [ ]:
# Build long-form (one row per participant × round) over vigilant participants
# who completed and passed the attention check — matches the filter used by the
# original R script that produced speaker_type.csv.
def make_speaker_type_long(df_anon):
    rows = []
    df_v = df_anon[
        (df_anon["listener_belief_condition"] == "vigilant")
        & (df_anon["completed_all_rounds"] == True)
        & (df_anon["attention_check_passed"] == True)
    ].copy()
    df_v["speaker_sequence"] = df_v["speaker_condition"].astype(str) + "_" + df_v["sequence_idx"].astype(int).astype(str)
    for r in range(1, 6):
        col = f"r{r}_speaker_type"
        sub = df_v[["participant_id", "speaker_sequence"]].copy()
        sub["round"] = r
        sub["speaker_type"] = df_v[col]
        rows.append(sub)
    long = pd.concat(rows, ignore_index=True)
    long = long[long["speaker_type"].notna() & (long["speaker_type"] != "") & (long["speaker_type"] != "null")]
    return long

# Aggregate counts → proportions, with explicit zero-fill for missing types.
def aggregate_speaker_type(long):
    SPEAKER_TYPES = ["anti", "neutral", "pro"]
    counts = long.groupby(["speaker_sequence", "round", "speaker_type"]).size().reset_index(name="n")
    # Build a complete grid (sequence × round × type) so missing types appear with n=0.
    seqs = sorted(long["speaker_sequence"].unique())
    grid = pd.MultiIndex.from_product(
        [seqs, range(1, 6), SPEAKER_TYPES],
        names=["speaker_sequence", "round", "speaker_type"],
    ).to_frame(index=False)
    out = grid.merge(counts, on=["speaker_sequence", "round", "speaker_type"], how="left").fillna({"n": 0})
    # Total per (sequence, round) = number of vigilant participants for that sequence.
    totals = out.groupby(["speaker_sequence", "round"])["n"].sum().reset_index(name="total")
    out = out.merge(totals, on=["speaker_sequence", "round"])
    # Match the R script's behavior: when total == 0, leave total as NA and percentage as 0.
    out["percentage"] = np.where(out["total"] > 0, out["n"] / out["total"] * 100.0, 0.0)
    out.loc[out["total"] == 0, "total"] = np.nan
    out["n"] = out["n"].astype(int)
    return out

long = make_speaker_type_long(df_anon)
spk = aggregate_speaker_type(long)
spk.to_csv("./speaker_type.csv", index=False)
print(f"Saved speaker_type.csv: {len(spk)} rows")
print(spk.head(9))


## Generate `data_dictionary.csv`

Per-column profile of the anonymized table (dtype, missingness, unique counts,
example values for low-cardinality categoricals).

In [ ]:
def profile_dataframe(df, key_cols=("participant_id",), max_levels=25):
    rows = []
    n = len(df)
    for col in df.columns:
        s = df[col]
        info = {
            "column": col,
            "dtype": str(s.dtype),
            "n_rows": n,
            "n_missing": int(s.isna().sum()),
            "pct_missing": float(s.isna().mean()),
            "n_unique": int(s.nunique(dropna=True)),
            "is_key": col in key_cols,
        }
        if pd.api.types.is_numeric_dtype(s) and info["n_missing"] < n:
            info.update(min=float(np.nanmin(s)), max=float(np.nanmax(s)),
                        mean=float(np.nanmean(s)), example_values="")
        else:
            info.update(min="", max="", mean="")
            uniq = s.dropna().astype(str).unique().tolist()
            info["example_values"] = "; ".join(uniq[:max_levels]) if len(uniq) <= max_levels else "; ".join(uniq[:5])
        rows.append(info)
    return pd.DataFrame(rows).sort_values(["is_key", "pct_missing", "n_unique"], ascending=[False, False, True])

dict_df = profile_dataframe(df_anon)
dict_df.to_csv("./data_dictionary.csv", index=False)
print(f"Saved data_dictionary.csv: {len(dict_df)} rows (one per column)")
